# Mission : Pilotez l'atterrisseur lunaire Eagle-1


## Installation des dépendances

Ce notebook est exécuté sur Google Colab la celulle suivante installe les dépendance nécessaires

In [1]:
!pip install gymnasium[box2d] stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 53.2 MB/s eta 0:00:00


## Création de l'environnement

In [2]:
import gymnasium as gym

# Create environment
env = gym.make("LunarLander-v3")
# env = gym.make("LunarLander-v3", render_mode="rgb_array")


print(f"Observation space: {env.observation_space}")
print(f"Observation shape: {env.observation_space.shape}")
print(f"Observation sample: {env.observation_space.sample()}")

print(f"Action space: {env.action_space}")
print(f"Action sample: {env.action_space.sample()}")
print(f"Nombre d'actions: {env.action_space.n}")

Observation space: Box([ -2.5        -2.5       -10.        -10.         -6.2831855 -10.
  -0.         -0.       ], [ 2.5        2.5       10.        10.         6.2831855 10.
  1.         1.       ], (8,), float32)
Observation shape: (8,)
Observation sample: [-2.2544973   0.02099045 -8.796453   -8.314228    4.0484266   1.419005
  0.4512928   0.50468826]
Action space: Discrete(4)
Action sample: 1
Nombre d'actions: 4


**Observation de l'environnement**

L'espace d'observation contient 8 variables continues : position (x, y),
vitesse (x, y), angle, vitesse angulaire, et 2 capteurs de contact (booléens).

L'espace d'action est discret avec 4 actions possibles :
- 0 : ne rien faire
- 1 : allumer le propulseur gauche
- 2 : allumer le propulseur principal (bas)
- 3 : allumer le propulseur droit

## Choix de l'algorithme

L'espace d'action de LunarLander-v3 est discret (4 actions), ce qui oriente le choix vers deux algorithmes compatibles avec Stable-Baselines3.

**DQN (Deep Q-Network)** apprend une fonction Q(s, a) qui estime la récompense future pour chaque action depuis un état donné. Il sélectionne toujours l'action avec la valeur Q la plus élevée et stocke ses expériences dans un replay buffer pour stabiliser l'apprentissage (off-policy).

**PPO (Proximal Policy Optimization)** apprend directement une politique : une distribution de probabilités sur les actions à prendre. Il ajoute une contrainte de clipping pour éviter des mises à jour trop brusques (on-policy). Bien que conçu pour les espaces continus, il fonctionne aussi sur du discret.

On part sur **DQN** pour cette mission : il est explicitement recommandé par le brief pour les espaces d'action discrets, et a déjà été appliqué sur CartPole en exercice 3 — le pipeline est connu.


In [3]:
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

# MlpPolicy = réseau de neurones dense (Multi-Layer Perceptron)
model = DQN('MlpPolicy', env, verbose=1, tensorboard_log="./logs/")

model.learn(total_timesteps=500000)

# Évaluation sur 50 épisodes pour avoir une mesure fiable
mean_reward, std_reward = evaluate_policy(model, env, n_eval_episodes=50)
print(f"Récompense moyenne : {mean_reward:.2f} +/- {std_reward:.2f}")

model.save("dqn_baseline")
print("Modèle sauvegardé : dqn_baseline.zip")

from google.colab import files
files.download("dqn_baseline.zip")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


Logging to ./logs/DQN_1


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 98       |
|    ep_rew_mean      | -98.3    |
|    exploration_rate | 0.993    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 881      |
|    time_elapsed     | 0        |
|    total_timesteps  | 392      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.935    |
|    n_updates        | 72       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 103      |
|    ep_rew_mean      | -146     |
|    exploration_rate | 0.984    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 1046     |
|    time_elapsed     | 0        |
|    total_timesteps  | 821      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.02     |
|    n_updates      

/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/evaluation.py:71: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


Récompense moyenne : 31.22 +/- 118.81
Modèle sauvegardé : dqn_baseline.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Observation :**

Entraînement : 500 000 timesteps, hyperparamètres par défaut.  
Récompense moyenne : 31.22 +/- 118.81 sur 50 épisodes.

L'écart-type élevé (118.81) révèle une forte variabilité : l'agent atterrit 
correctement sur certains épisodes mais s'écrase ou sort des limites sur d'autres.

Le score moyen (31.22) reste loin de l'objectif de 200, l'optimisation des 
hyperparamètres est nécessaire.

